[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Barnabas4444/ProjectGrazingCow/blob/main/pasture_biomass_tensorflow.ipynb)

In [23]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

DATA_DIR = Path.cwd()
IMAGE_SIZE = 320
BATCH_SIZE = 16
SEED = 42
TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
TARGET_WEIGHTS = tf.constant([0.1, 0.1, 0.1, 0.2, 0.5], dtype=tf.float32)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU devices: {tf.config.list_physical_devices("GPU")}')
print(f'Data directory: {DATA_DIR}')
print(f'Training resolution: {IMAGE_SIZE}px | Batch size: {BATCH_SIZE}')

train_csv = pd.read_csv(DATA_DIR / 'train.csv')
test_csv = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Training CSV shape: {train_csv.shape}')
print(f'Test CSV shape: {test_csv.shape}')
print('Raw training rows (long format):')
display(train_csv.head(10))
print('Test rows:')
display(test_csv.head(10))

TensorFlow version: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Data directory: /mnt/d/BME/Deep Learning feleves
Training resolution: 320px | Batch size: 16
Training CSV shape: (1785, 9)
Test CSV shape: (5, 3)
Raw training rows (long format):


,sample_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,target_name,target
0,ID1011485656__Dry_Clover_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Clover_g,0.0000
1,ID1011485656__Dry_Dead_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Dead_g,31.9984
2,ID1011485656__Dry_Green_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Green_g,16.2751
3,ID1011485656__Dry_Total_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Total_g,48.2735
4,ID1011485656__GDM_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,GDM_g,16.2750
5,ID1012260530__Dry_Clover_g,train/ID1012260530.jpg,2015/4/1,NSW,Lucerne,0.55,16.0000,Dry_Clover_g,0.0000
6,ID1012260530__Dry_Dead_g,train/ID1012260530.jpg,2015/4/1,NSW,Lucerne,0.55,16.0000,Dry_Dead_g,0.0000
7,ID1012260530__Dry_Green_g,train/ID1012260530.jpg,2015/4/1,NSW,Lucerne,0.55,16.0000,Dry_Green_g,7.6000
8,ID1012260530__Dry_Total_g,train/ID1012260530.jpg,2015/4/1,NSW,Lucerne,0.55,16.0000,Dry_Total_g,7.6000
9,ID1012260530__GDM_g,train/ID1012260530.jpg,2015/4/1,NSW,Lucerne,0.55,16.0000,GDM_g,7.6000


Test rows:


,sample_id,image_path,target_name
0,ID1001187975__Dry_Clover_g,test/ID1001187975.jpg,Dry_Clover_g
1,ID1001187975__Dry_Dead_g,test/ID1001187975.jpg,Dry_Dead_g
2,ID1001187975__Dry_Green_g,test/ID1001187975.jpg,Dry_Green_g
3,ID1001187975__Dry_Total_g,test/ID1001187975.jpg,Dry_Total_g
4,ID1001187975__GDM_g,test/ID1001187975.jpg,GDM_g


In [24]:
train_wide = train_csv.pivot(
    index='image_path',
    columns='target_name',
    values='target',
).reset_index()

missing_targets = set(TARGETS) - set(train_wide.columns)
if missing_targets:
    raise ValueError(f'Missing training targets: {missing_targets}')

train_wide = train_wide[['image_path'] + TARGETS]

print(f'Wide training table shape: {train_wide.shape}')
print('One row per image, with its five target values:')
display(train_wide.head(5))

Wide training table shape: (357, 6)
One row per image, with its five target values:


target_name,image_path,Dry_Green_g,Dry_Dead_g,Dry_Clover_g,GDM_g,Dry_Total_g
0,train/ID1011485656.jpg,16.2751,31.9984,0.0000,16.2750,48.2735
1,train/ID1012260530.jpg,7.6000,0.0000,0.0000,7.6000,7.6000
2,train/ID1025234388.jpg,0.0000,0.0000,6.0500,6.0500,6.0500
3,train/ID1028611175.jpg,24.2376,30.9703,0.0000,24.2376,55.2079
4,train/ID1035947949.jpg,10.5261,23.2239,0.4343,10.9605,34.1844


In [25]:
def resolve_paths(relative_paths):
    paths = [str(DATA_DIR / path) for path in relative_paths]
    missing = [path for path in paths if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError(f'{len(missing)} image(s) missing; first: {missing[0]}')
    return np.asarray(paths)

image_paths = resolve_paths(train_wide['image_path'].tolist())
targets = train_wide[TARGETS].to_numpy(dtype=np.float32)
train_paths, val_paths, y_train, y_val = train_test_split(
    image_paths, targets, test_size=0.2, random_state=SEED
)

print(f'Train images: {len(train_paths)} | Validation images: {len(val_paths)}')
print(f'Target array shape: {targets.shape} (images, targets)')
print('Target column order:', TARGETS)
print('Example training label vector:', dict(zip(TARGETS, y_train[0])))

Train images: 285 | Validation images: 72
Target array shape: (357, 5) (images, targets)
Target column order: ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
Example training label vector: {'Dry_Green_g': np.float32(36.9586), 'Dry_Dead_g': np.float32(8.0345), 'Dry_Clover_g': np.float32(1.6069), 'GDM_g': np.float32(38.5655), 'Dry_Total_g': np.float32(46.6)}


In [26]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize_with_pad(image, IMAGE_SIZE, IMAGE_SIZE)
    return tf.cast(image, tf.float32)

def make_dataset(paths, labels=None, training=False):
    if labels is None:
        dataset = tf.data.Dataset.from_tensor_slices(paths)
        dataset = dataset.map(lambda path: {'image': load_image(path)}, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
        dataset = dataset.map(
            lambda path, label: ({'image': load_image(path)}, label),
            num_parallel_calls=tf.data.AUTOTUNE,
        )
    if training:
        dataset = dataset.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_paths, y_train, training=True)
val_ds = make_dataset(val_paths, y_val)

In [27]:
def weighted_mse(y_true, y_pred):
    squared_error = tf.square(y_true - y_pred)
    return tf.reduce_sum(squared_error * TARGET_WEIGHTS, axis=-1) / tf.reduce_sum(TARGET_WEIGHTS)

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.06),
    tf.keras.layers.RandomZoom(0.08),
], name='augmentation')

image_input = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name='image')
x = augmentation(image_input)
backbone = tf.keras.applications.EfficientNetB0(
    include_top=False, weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
)
backbone.trainable = False
x = backbone(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.2)(x)
output = tf.keras.layers.Dense(len(TARGETS), name='biomass')(x)
model = tf.keras.Model(inputs=image_input, outputs=output)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=weighted_mse)
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-6),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=callbacks)

Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: image
Received: inputs=['Tensor(shape=(None, 320, 320, 3))']
  warnings.warn(msg)


18/18 ━━━━━━━━━━━━━━━━━━━━ 9s 184ms/step - loss: 1341.8967 - val_loss: 657.5054 - learning_rate: 0.0010
Epoch 2/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 111ms/step - loss: 668.2761 - val_loss: 428.1017 - learning_rate: 0.0010
Epoch 3/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 114ms/step - loss: 556.3123 - val_loss: 416.0631 - learning_rate: 0.0010
Epoch 4/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 110ms/step - loss: 510.1609 - val_loss: 380.4126 - learning_rate: 0.0010
Epoch 5/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 111ms/step - loss: 495.3215 - val_loss: 359.7949 - learning_rate: 0.0010
Epoch 6/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 106ms/step - loss: 476.8601 - val_loss: 351.2069 - learning_rate: 0.0010
Epoch 7/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 105ms/step - loss: 427.4061 - val_loss: 366.1427 - learning_rate: 0.0010
Epoch 8/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 107ms/step - loss: 435.0325 - val_loss: 345.5849 - learning_rate: 0.0010
Epoch 9/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 105ms/step - loss: 439.2947 - val_loss: 345.7058 - learni

In [28]:
best_frozen_weights = model.get_weights()
frozen_val_loss = model.evaluate(val_ds, verbose=0)
backbone.trainable = True
for layer in backbone.layers[:-30]:
    layer.trainable = False
for layer in backbone.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss=weighted_mse)
fine_tune_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-7),
]
model.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=fine_tune_callbacks)
fine_tuned_val_loss = model.evaluate(val_ds, verbose=0)
if fine_tuned_val_loss > frozen_val_loss:
    model.set_weights(best_frozen_weights)
    print('Fine-tuning did not improve validation loss; keeping frozen-backbone weights.')

val_predictions = model.predict(val_ds, verbose=0)
y_true = y_val.reshape(-1)
y_pred = val_predictions.reshape(-1)
row_weights = np.tile(np.array([0.1, 0.1, 0.1, 0.2, 0.5]), len(y_val))
weighted_mean = np.sum(row_weights * y_true) / np.sum(row_weights)
rss = np.sum(row_weights * np.square(y_true - y_pred))
tss = np.sum(row_weights * np.square(y_true - weighted_mean))
print(f'Validation weighted R^2: {1.0 - rss / tss:.5f}')

Epoch 1/12


E0000 00:00:1791992124.718082    6954 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_20_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


18/18 ━━━━━━━━━━━━━━━━━━━━ 11s 187ms/step - loss: 374.7801 - val_loss: 281.4324 - learning_rate: 1.0000e-05
Epoch 2/12
18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 117ms/step - loss: 345.3707 - val_loss: 296.9543 - learning_rate: 1.0000e-05
Epoch 3/12
18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 114ms/step - loss: 351.3219 - val_loss: 296.0729 - learning_rate: 1.0000e-05
Epoch 4/12
18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 118ms/step - loss: 367.6124 - val_loss: 293.3476 - learning_rate: 3.0000e-06
Epoch 5/12
18/18 ━━━━━━━━━━━━━━━━━━━━ 4s 117ms/step - loss: 351.3209 - val_loss: 287.7542 - learning_rate: 3.0000e-06


/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: image
Received: inputs=['Tensor(shape=(16, 320, 320, 3))']
  warnings.warn(msg)


Validation weighted R^2: 0.55850


In [29]:
def validation_weighted_r2(y_true_values, y_pred_values):
    true_values = y_true_values.reshape(-1)
    predicted_values = y_pred_values.reshape(-1)
    weights = np.tile(np.array([0.1, 0.1, 0.1, 0.2, 0.5]), len(y_true_values))
    weighted_mean = np.sum(weights * true_values) / np.sum(weights)
    residual_sum = np.sum(weights * np.square(true_values - predicted_values))
    total_sum = np.sum(weights * np.square(true_values - weighted_mean))
    return 1.0 - residual_sum / total_sum

baseline_r2 = validation_weighted_r2(y_val, val_predictions)
derived_gdm_predictions = val_predictions.copy()
derived_gdm_predictions[:, TARGETS.index('GDM_g')] = (
    derived_gdm_predictions[:, TARGETS.index('Dry_Green_g')]
    + derived_gdm_predictions[:, TARGETS.index('Dry_Clover_g')]
)
derived_gdm_r2 = validation_weighted_r2(y_val, derived_gdm_predictions)

print(f'Independent GDM weighted R^2: {baseline_r2:.5f}')
print(f'Derived GDM weighted R^2: {derived_gdm_r2:.5f}')

Independent GDM weighted R^2: 0.55850
Derived GDM weighted R^2: 0.56330


In [30]:
def derive_gdm(predictions):
    predictions = predictions.copy()
    predictions[:, TARGETS.index('GDM_g')] = (
        predictions[:, TARGETS.index('Dry_Green_g')]
        + predictions[:, TARGETS.index('Dry_Clover_g')]
    )
    return predictions

class BestWeightedR2(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        predictions = derive_gdm(self.model.predict(val_ds, verbose=0))
        self.initial_score = validation_weighted_r2(y_val, predictions)
        self.best_score = self.initial_score
        self.best_weights = self.model.get_weights()
        print(f'Starting weighted R^2: {self.initial_score:.5f}')

    def on_epoch_end(self, epoch, logs=None):
        predictions = derive_gdm(self.model.predict(val_ds, verbose=0))
        score = validation_weighted_r2(y_val, predictions)
        logs['val_weighted_r2'] = score
        if score > self.best_score:
            self.best_score = score
            self.best_weights = self.model.get_weights()
        print(f'Validation weighted R^2: {score:.5f}; best: {self.best_score:.5f}')

    def on_train_end(self, logs=None):
        self.model.set_weights(self.best_weights)

backbone.trainable = True
for layer in backbone.layers:
    layer.trainable = False
for layer in backbone.layers[-80:]:
    layer.trainable = not isinstance(layer, tf.keras.layers.BatchNormalization)

model.compile(optimizer=tf.keras.optimizers.Adam(3e-5), loss=weighted_mse)
score_checkpoint = BestWeightedR2()
improved_history = model.fit(
    train_ds, validation_data=val_ds, epochs=25,
    callbacks=[
        score_checkpoint,
        tf.keras.callbacks.EarlyStopping(
            monitor='val_weighted_r2', mode='max', patience=8
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_weighted_r2', mode='max', factor=0.3,
            patience=3, min_lr=1e-7
        ),
    ],
)
val_predictions = model.predict(val_ds, verbose=0)
selected_predictions = derive_gdm(val_predictions)
selected_r2 = validation_weighted_r2(y_val, selected_predictions)
assert selected_r2 >= score_checkpoint.initial_score - 1e-6
print(f'Baseline weighted R^2: {score_checkpoint.initial_score:.5f}')
print(f'Selected weighted R^2: {selected_r2:.5f}')

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: image
Received: inputs=['Tensor(shape=(16, 320, 320, 3))']
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: image
Received: inputs=['Tensor(shape=(None, 320, 320, 3))']
  warnings.warn(msg)


Starting weighted R^2: 0.56330
Epoch 1/25


E0000 00:00:1791992207.451654    6954 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_20_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - loss: 353.5555Validation weighted R^2: 0.50877; best: 0.56330
18/18 ━━━━━━━━━━━━━━━━━━━━ 12s 247ms/step - loss: 353.5555 - val_loss: 319.7921 - val_weighted_r2: 0.5088 - learning_rate: 3.0000e-05
Epoch 2/25
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - loss: 353.5703Validation weighted R^2: 0.57934; best: 0.57934
18/18 ━━━━━━━━━━━━━━━━━━━━ 5s 176ms/step - loss: 353.5703 - val_loss: 271.4516 - val_weighted_r2: 0.5793 - learning_rate: 3.0000e-05
Epoch 3/25
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - loss: 299.2278Validation weighted R^2: 0.59432; best: 0.59432
18/18 ━━━━━━━━━━━━━━━━━━━━ 5s 179ms/step - loss: 299.2278 - val_loss: 260.0505 - val_weighted_r2: 0.5943 - learning_rate: 3.0000e-05
Epoch 4/25
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - loss: 312.6978Validation weighted R^2: 0.58331; best: 0.59432
18/18 ━━━━━━━━━━━━━━━━━━━━ 5s 178ms/step - loss: 312.6978 - val_loss: 265.4002 - val_weighted_r2: 0.5833 - learning_rate: 3.0000e-05
Epoch 5/25
18/

In [31]:
continuation_baseline_predictions = np.maximum(
    derive_gdm(model.predict(val_ds, verbose=0)), 0.0
)
continuation_baseline_r2 = validation_weighted_r2(y_val, continuation_baseline_predictions)
continuation_baseline_weights = model.get_weights()

backbone.trainable = True
for layer in backbone.layers:
    layer.trainable = False
for layer in backbone.layers[-120:]:
    layer.trainable = not isinstance(layer, tf.keras.layers.BatchNormalization)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss=weighted_mse)
continuation_checkpoint = BestWeightedR2()
continuation_history = model.fit(
    train_ds, validation_data=val_ds, epochs=20,
    callbacks=[
        continuation_checkpoint,
        tf.keras.callbacks.EarlyStopping(
            monitor='val_weighted_r2', mode='max', patience=6
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_weighted_r2', mode='max', factor=0.3,
            patience=3, min_lr=1e-7
        ),
    ],
)
val_predictions = model.predict(val_ds, verbose=0)
continuation_r2 = validation_weighted_r2(
    y_val, np.maximum(derive_gdm(val_predictions), 0.0)
)
if continuation_r2 < continuation_baseline_r2:
    model.set_weights(continuation_baseline_weights)
    val_predictions = model.predict(val_ds, verbose=0)
selected_r2 = validation_weighted_r2(y_val, np.maximum(derive_gdm(val_predictions), 0.0))
assert selected_r2 >= continuation_baseline_r2 - 1e-6
print(f'Previous submission-style weighted R^2: {continuation_baseline_r2:.5f}')
print(f'Selected submission-style weighted R^2: {selected_r2:.5f}')

Starting weighted R^2: 0.60822
Epoch 1/20


E0000 00:00:1791992296.682260    6954 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_20_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - loss: 280.3652Validation weighted R^2: 0.57352; best: 0.60822
18/18 ━━━━━━━━━━━━━━━━━━━━ 14s 259ms/step - loss: 280.3652 - val_loss: 277.2921 - val_weighted_r2: 0.5735 - learning_rate: 1.0000e-05
Epoch 2/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - loss: 265.6035Validation weighted R^2: 0.61926; best: 0.61926
18/18 ━━━━━━━━━━━━━━━━━━━━ 5s 202ms/step - loss: 265.6035 - val_loss: 244.5789 - val_weighted_r2: 0.6193 - learning_rate: 1.0000e-05
Epoch 3/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - loss: 282.7644Validation weighted R^2: 0.59921; best: 0.61926
18/18 ━━━━━━━━━━━━━━━━━━━━ 5s 195ms/step - loss: 282.7644 - val_loss: 259.6146 - val_weighted_r2: 0.5992 - learning_rate: 1.0000e-05
Epoch 4/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - loss: 263.2729Validation weighted R^2: 0.61244; best: 0.61926
18/18 ━━━━━━━━━━━━━━━━━━━━ 5s 201ms/step - loss: 263.2729 - val_loss: 250.5365 - val_weighted_r2: 0.6124 - learning_rate: 1.0000e-05
Epoch 5/20
18/

In [32]:
polish_baseline_weights = model.get_weights()
polish_baseline_predictions = np.maximum(
    derive_gdm(model.predict(val_ds, verbose=0)), 0.0
)
polish_baseline_r2 = validation_weighted_r2(y_val, polish_baseline_predictions)
model.compile(optimizer=tf.keras.optimizers.Adam(3e-6), loss=weighted_mse)
polish_checkpoint = BestWeightedR2()
polish_history = model.fit(
    train_ds, validation_data=val_ds, epochs=20,
    callbacks=[
        polish_checkpoint,
        tf.keras.callbacks.EarlyStopping(
            monitor='val_weighted_r2', mode='max', patience=6
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_weighted_r2', mode='max', factor=0.3,
            patience=3, min_lr=1e-7
        ),
    ],
)
val_predictions = model.predict(val_ds, verbose=0)
polish_r2 = validation_weighted_r2(
    y_val, np.maximum(derive_gdm(val_predictions), 0.0)
)
if polish_r2 < polish_baseline_r2:
    model.set_weights(polish_baseline_weights)
    val_predictions = model.predict(val_ds, verbose=0)
selected_r2 = validation_weighted_r2(y_val, np.maximum(derive_gdm(val_predictions), 0.0))
assert selected_r2 >= polish_baseline_r2 - 1e-6
print(f'Before lower-rate training R^2: {polish_baseline_r2:.5f}')
print(f'Selected weighted R^2: {selected_r2:.5f}')

Starting weighted R^2: 0.62416
Epoch 1/20


E0000 00:00:1791992371.734669    6954 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_20_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - loss: 255.1384Validation weighted R^2: 0.61950; best: 0.62416
18/18 ━━━━━━━━━━━━━━━━━━━━ 13s 255ms/step - loss: 255.1384 - val_loss: 245.8072 - val_weighted_r2: 0.6195 - learning_rate: 3.0000e-06
Epoch 2/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - loss: 239.3375Validation weighted R^2: 0.62037; best: 0.62416
18/18 ━━━━━━━━━━━━━━━━━━━━ 5s 197ms/step - loss: 239.3375 - val_loss: 245.4993 - val_weighted_r2: 0.6204 - learning_rate: 3.0000e-06
Epoch 3/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - loss: 233.6959Validation weighted R^2: 0.61402; best: 0.62416
18/18 ━━━━━━━━━━━━━━━━━━━━ 5s 197ms/step - loss: 233.6959 - val_loss: 249.8585 - val_weighted_r2: 0.6140 - learning_rate: 3.0000e-06
Epoch 4/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - loss: 248.2248Validation weighted R^2: 0.61688; best: 0.62416
18/18 ━━━━━━━━━━━━━━━━━━━━ 5s 198ms/step - loss: 248.2248 - val_loss: 247.6653 - val_weighted_r2: 0.6169 - learning_rate: 3.0000e-06
Epoch 5/20
18/

In [33]:
from scipy.optimize import nnls

relation_matrix = np.array([
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 0.0],
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 1.0],
])
relation_scale = np.sqrt(np.array([0.1, 0.1, 0.1, 0.2, 0.5]))

def reconcile_predictions(predictions):
    components = np.array([
        nnls(relation_matrix * relation_scale[:, None], prediction * relation_scale)[0]
        for prediction in predictions
    ])
    return components @ relation_matrix.T

reconciliation_raw_predictions = model.predict(val_ds, verbose=0)
reconciliation_baseline = np.maximum(derive_gdm(reconciliation_raw_predictions), 0.0)
reconciliation_predictions = reconcile_predictions(reconciliation_raw_predictions)
reconciliation_baseline_r2 = validation_weighted_r2(y_val, reconciliation_baseline)
reconciliation_r2 = validation_weighted_r2(y_val, reconciliation_predictions)
use_relation_projection = reconciliation_r2 > reconciliation_baseline_r2
np.testing.assert_allclose(
    reconciliation_predictions[:, 3],
    reconciliation_predictions[:, 0] + reconciliation_predictions[:, 2],
)
assert (reconciliation_predictions >= 0).all()
print(f'Derived GDM + clipping weighted R^2: {reconciliation_baseline_r2:.5f}')
print(f'Nonnegative reconciled weighted R^2: {reconciliation_r2:.5f}')
print(f'Use reconciled predictions: {use_relation_projection}')

Derived GDM + clipping weighted R^2: 0.62604
Nonnegative reconciled weighted R^2: 0.62113
Use reconciled predictions: False


In [34]:
def predict_views(dataset, horizontal_flip=False):
    batches = []
    for batch in dataset:
        inputs = batch[0] if isinstance(batch, tuple) else batch
        images = inputs['image']
        predictions = model({'image': images}, training=False).numpy()
        if horizontal_flip:
            flipped_predictions = model(
                {'image': tf.image.flip_left_right(images)}, training=False
            ).numpy()
            predictions = (predictions + flipped_predictions) / 2.0
        batches.append(predictions)
    return np.maximum(derive_gdm(np.concatenate(batches, axis=0)), 0.0)

single_view_predictions = predict_views(val_ds)
flip_view_predictions = predict_views(val_ds, horizontal_flip=True)
single_view_r2 = validation_weighted_r2(y_val, single_view_predictions)
flip_view_r2 = validation_weighted_r2(y_val, flip_view_predictions)
use_horizontal_flip = flip_view_r2 > single_view_r2
inference_predictions = flip_view_predictions if use_horizontal_flip else single_view_predictions
inference_r2 = validation_weighted_r2(y_val, inference_predictions)
assert np.isfinite(inference_predictions).all()
assert inference_predictions.shape == y_val.shape
print(f'Single-view submission-style weighted R^2: {single_view_r2:.5f}')
print(f'Original + horizontal flip weighted R^2: {flip_view_r2:.5f}')
print(f'Use horizontal-flip averaging: {use_horizontal_flip}')

/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: image
Received: inputs=['Tensor(shape=(8, 320, 320, 3))']
  warnings.warn(msg)


Single-view submission-style weighted R^2: 0.62606
Original + horizontal flip weighted R^2: 0.61542
Use horizontal-flip averaging: False


In [35]:
test_images = test_csv['image_path'].drop_duplicates().tolist()
test_paths = resolve_paths(test_images)
test_ds = make_dataset(test_paths)
test_predictions = predict_views(test_ds, horizontal_flip=use_horizontal_flip)
prediction_by_image = dict(zip(test_images, test_predictions))
target_index = {name: index for index, name in enumerate(TARGETS)}

submission = test_csv[['sample_id']].copy()
submission['target'] = [
    float(prediction_by_image[image_path][target_index[target_name]])
    for image_path, target_name in zip(test_csv['image_path'], test_csv['target_name'])
]
assert submission['sample_id'].equals(test_csv['sample_id'])
assert np.isfinite(submission['target']).all()
assert (submission['target'] >= 0).all()
assert inference_r2 >= continuation_baseline_r2 - 1e-5
output_path = DATA_DIR / 'submission.csv'
submission.to_csv(output_path, index=False)
weights_path = DATA_DIR / 'pasture_best.weights.h5'
model.save_weights(weights_path)
print(submission.head(10))
print(f'Selected inference weighted R^2: {inference_r2:.5f}')
print(f'Saved {output_path} with {len(submission)} rows.')
print(f'Saved selected model weights to {weights_path}')

/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: image
Received: inputs=['Tensor(shape=(1, 320, 320, 3))']
  warnings.warn(msg)


                    sample_id     target
0  ID1001187975__Dry_Clover_g   0.000000
1    ID1001187975__Dry_Dead_g  19.784653
2   ID1001187975__Dry_Green_g  43.727566
3   ID1001187975__Dry_Total_g  61.312740
4         ID1001187975__GDM_g  39.401978
Selected inference weighted R^2: 0.62606
Saved /mnt/d/BME/Deep Learning feleves/submission.csv with 5 rows.
Saved selected model weights to /mnt/d/BME/Deep Learning feleves/pasture_best.weights.h5
